In [1]:
import os
import sys
import re
import collections

import joblib
import numpy as np
import pandas as pd

# VSCode's Jupyter kernel doesn't reliably start with CWD = this notebook's own
# directory (and a stray chdir elsewhere in the session can also leave it wrong) --
# anchor off the notebook's own absolute path instead of trusting os.getcwd().
try:
    _nb = globals().get('__vsc_ipynb_file__')
    if _nb:
        os.chdir(os.path.dirname(_nb))
except Exception:
    pass

_MAIN_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, _MAIN_DIR)
sys.path.insert(0, os.path.join(_MAIN_DIR, "utils"))
sys.path.insert(0, os.path.join(_MAIN_DIR, "utils", "model_training"))

import config

GROUP_NAME = "final_6_new"
EXP_FOLDER = "/vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final"
DATASETS = config.CROSS_DATASET_GROUPS[GROUP_NAME]
PC_WELL_IDX = 8  # every chip's LABEL_MAPPINGS assigns well 8 to PC

# Paper-facing target names, matching the methodology chapter's own wording
# ("five targets (IAV, IBV, KP, SARS-CoV-2 and hAdV)") rather than the code's
# internal shorthand (Kp/Cov/Hadv).
TARGET_DISPLAY_NAME = {
    "IAV": "IAV", "IBV": "IBV", "Kp": "KP", "Cov": "SARS-CoV-2", "Hadv": "hAdV",
}
TARGET_ORDER = ["IAV", "IBV", "KP", "SARS-CoV-2", "hAdV"]
CONC_ORDER = [10_000, 100_000, 1_000_000]
CONC_DISPLAY = {10_000: "$10^4$", 100_000: "$10^5$", 1_000_000: "$10^6$"}


def chip_label(dataset_name):
    """'D20260806_E00_C00_F4500KHz_U_DDM_01_06' -> 'DDM_01'"""
    m = re.search(r"DDM_(\d+)", dataset_name)
    return f"DDM_{m.group(1)}"



[*] SUCCESS: TensorFlow is utilizing the GPU -> /physical_device:GPU:0



## Load per-well curve counts

For each chip: count curves per well index in the main `Y_well` array (targets + `NC-ALL`), skipping well 8 (PC) there since it's always empty post-`--drop_pc`; then add the real PC count from the separately-stored `pc_wells` snapshot.

In [2]:
def load_well_counts(dataset_name):
    """Returns a list of (well_idx, label, concentration, n_curves) for one chip:
    the main Y_well wells (targets + NC-ALL) plus the separately-stored PC well."""
    data_path = os.path.join(EXP_FOLDER, dataset_name, config.TRAINING_DATA_PATH)
    d = joblib.load(data_path)

    label_map = config.LABEL_MAPPINGS.get(dataset_name, {})
    conc_map = config.CONC_MAPPINGS.get(dataset_name, {})

    Y_well = np.asarray(d["Y_well"])
    counts = collections.Counter(Y_well.tolist())

    rows = []
    for well_idx, label in label_map.items():
        if well_idx == PC_WELL_IDX:
            continue  # real PC count comes from pc_wells below, not main Y_well
        n = counts.get(well_idx, 0)
        rows.append((well_idx, label, conc_map.get(well_idx), n))

    # Positive control: dropped from Y_well by --drop_pc, snapshotted separately.
    pc = d.get("pc_wells")
    if pc and "Y_well" in pc:
        n_pc = len(pc["Y_well"])
    elif pc and "curves" in pc:
        first_variant = next(iter(pc["curves"].values()))
        n_pc = np.asarray(first_variant).shape[0]
    else:
        n_pc = 0
    rows.append((PC_WELL_IDX, "PC", 0, n_pc))

    return rows


records = []
for name in DATASETS:
    for well_idx, label, conc, n in load_well_counts(name):
        records.append({
            "chip": chip_label(name),
            "well": well_idx,
            "label": label,
            "concentration": conc,
            "n_curves": n,
        })

well_df = pd.DataFrame.from_records(records)
well_df

,chip,well,label,concentration,n_curves
0,DDM_01,0,IAV,1000000,1942
1,DDM_01,1,IAV,100000,1871
2,DDM_01,2,IAV,10000,1914
3,DDM_01,3,IBV,10000,1807
4,DDM_01,4,Kp,100000,1713
5,DDM_01,5,Cov,1000000,1894
6,DDM_01,6,Hadv,1000000,1874
7,DDM_01,7,Hadv,100000,1680
8,DDM_01,9,NC-ALL,0,1686
9,DDM_01,8,PC,0,1828


## Pivot into the profile table

Rows = chip. Columns = each target split by concentration, plus **PC**, **NC**, and a **Total** column (sum of every well on that chip, including PC and NC). This wide, chip-per-row form is transposed below into the layout actually used for the LaTeX table.

In [3]:
def build_profile_table(well_df):
    df = well_df.copy()
    df["target"] = df["label"].map(TARGET_DISPLAY_NAME)

    target_rows = df[df["target"].notna()]
    wide = target_rows.pivot_table(
        index="chip", columns=["target", "concentration"],
        values="n_curves", aggfunc="sum", fill_value=0,
    )

    col_order = pd.MultiIndex.from_product([TARGET_ORDER, CONC_ORDER], names=["target", "concentration"])
    wide = wide.reindex(columns=col_order, fill_value=0)

    pc = df[df["label"] == "PC"].groupby("chip")["n_curves"].sum()
    nc = df[df["label"] == "NC-ALL"].groupby("chip")["n_curves"].sum()

    wide[("PC", "")] = pc.reindex(wide.index, fill_value=0)
    wide[("NC", "")] = nc.reindex(wide.index, fill_value=0)
    wide[("Total", "")] = wide.sum(axis=1)

    wide = wide.reindex(index=[chip_label(n) for n in DATASETS])
    wide.columns = pd.MultiIndex.from_tuples(wide.columns, names=["target", "concentration"])
    return wide


profile = build_profile_table(well_df)
profile

target          IAV                  IBV                   KP                 \
concentration 10000 100000 1000000 10000 100000 1000000 10000 100000 1000000   
chip                                                                           
DDM_01         1914   1871    1942  1807      0       0     0   1713       0   
DDM_02            0   1930    1949  1918   1881    1924  1756      0       0   
DDM_03         1946      0       0     0   1872    1892  1868   1668    1904   
DDM_04            0      0    1893  1852      0       0     0   1678    1825   
DDM_05            0   1955       0     0      0    1919  1943      0       0   
DDM_06         1890      0       0  1827   1936       0     0      0    1864   

target        SARS-CoV-2                 hAdV                   PC    NC  \
concentration      10000 100000 1000000 10000 100000 1000000               
chip                                                                       
DDM_01                 0      0    1894     0   1680    1874  1828  1686   
DDM_02                 0   1732       0  1720      0       0  1865  1828   
DDM_03              1876      0       0     0      0    1886  1869  1842   
DDM_04              1838   1932    1936     0   1752       0  1854  1774   
DDM_05                 0   1921    1884  1903   1945    1979  1896  1901   
DDM_06              1941   1916       0  3787      0       0  1715  1810   

target         Total  
concentration         
chip                  
DDM_01         18209  
DDM_02         18503  
DDM_03         18623  
DDM_04         18334  
DDM_05         19246  
DDM_06         18686

## Transposed view (target x concentration as rows, chip as columns)

With only 4 chips but 15 target/concentration combinations, a tall table (rows = target x concentration, plus PC/NC/Total; columns = chip) reads far better than the 18-column wide form above, and is what the LaTeX export below produces.

In [4]:
row_order = pd.MultiIndex.from_tuples(
    [(t, c) for t in TARGET_ORDER for c in CONC_ORDER] + [("PC", ""), ("NC", ""), ("Total", "")],
    names=["target", "concentration"],
)
profile_T = profile.T.reindex(row_order)

display_index = pd.MultiIndex.from_tuples(
    [(t, CONC_DISPLAY.get(c, c) if c != "" else "") for (t, c) in profile_T.index],
    names=["target", "concentration"],
)
profile_T_display = profile_T.copy()
profile_T_display.index = display_index
profile_T_display.style.format("{:,}")

## LaTeX table (transposed)

Booktabs style, matching `tab:notation` / `tab:lab_datasets` in the methodology chapter. Rows = target x concentration (`\multirow` groups each target's three concentrations) plus dedicated **PC** and **NC** rows and a **Total** row; columns = chip. Ready to drop into `03_methodology.tex` under \S\ref{app:chip_mapping} or the Lacewing eLAMP Platform subsection.

In [5]:
def build_lofo_excluded_cells(group_name):
    """{(chip_label, row_label, concentration)} for every well
    config.LOFO_EXCLUDE_WELL_MAPPING drops for this LOFO group -- PC/NC use
    concentration="" to match how build_profile_table keys those two rows."""
    excluded = set()
    mapping = config.LOFO_EXCLUDE_WELL_MAPPING.get(group_name) or {}
    for dataset_name, wells in mapping.items():
        label_map = config.LABEL_MAPPINGS.get(dataset_name, {})
        conc_map = config.CONC_MAPPINGS.get(dataset_name, {})
        chip = chip_label(dataset_name)
        for w in wells:
            raw_label = label_map.get(w)
            if raw_label is None:
                continue
            if raw_label == "PC":
                excluded.add((chip, "PC", ""))
            elif raw_label == "NC-ALL":
                excluded.add((chip, "NC", ""))
            else:
                target = TARGET_DISPLAY_NAME.get(raw_label, raw_label)
                excluded.add((chip, target, conc_map.get(w)))
    return excluded


_CHIP_DISPLAY_RENAME = ("DDM_0", "Chip 0")  # matches 4_visualise_recon_layer.ipynb's chip_title()


def chip_display(chip):
    return chip.replace(*_CHIP_DISPLAY_RENAME)


_CONTROL_DISPLAY = {"PC": "Positive Control", "NC": "Negative Control"}

_FOOTNOTE_SPECIAL = "$^*$This sample was loaded across two separate reaction wells."
_FOOTNOTE_FAILED = "$^{**}$This sample failed to produce usable signals."


def to_latex_profile_table_transposed(profile, caption, label, failed_cells=None,
                                      special_notes=None, n_timesteps=None, placement="htbp"):
    """failed_cells: {(chip, row_label, conc)} shown as "--$^{**}$" and left out of Total
    (non-PC/NC wells excluded from LOFO training). special_notes: {(chip, row_label, conc):
    marker} appended to the real value, e.g. "$^*$". n_timesteps: {chip: T} for an optional
    row below Total. PC/NC are never marked either way, regardless of exclusion status."""
    failed_cells = failed_cells or set()
    special_notes = special_notes or {}
    chips = list(profile.index)
    n_chips = len(chips)

    def fmt(chip, row_label, conc, v):
        key = (chip, row_label, conc)
        if key in failed_cells:
            return "--$^{**}$"
        s = f"{v:,}" if v else "--"
        return s + special_notes.get(key, "")

    col_spec = "ll" + "r" * n_chips
    lines = []
    lines.append(f"\\begin{{table}}[{placement}]")
    lines.append("    \\centering")
    lines.append(f"    \\caption{{{caption}}}")
    lines.append(f"    \\label{{{label}}}")
    lines.append("    \\small")
    lines.append(f"    \\begin{{tabular}}{{@{{}}{col_spec}@{{}}}}")
    lines.append("    \\toprule")

    header = ["\\textbf{Target}", "\\textbf{Conc.}"] + [f"\\textbf{{{chip_display(c)}}}" for c in chips]
    lines.append("    " + " & ".join(header) + " \\\\")
    lines.append("    \\midrule")

    def get(chip, col):
        return profile.loc[chip, col] if col in profile.columns else 0

    for t in TARGET_ORDER:
        for i, c in enumerate(CONC_ORDER):
            target_cell = f"\\multirow{{{len(CONC_ORDER)}}}{{*}}{{{t}}}" if i == 0 else ""
            cells = [target_cell, CONC_DISPLAY[c]]
            for chip in chips:
                v = get(chip, (t, c))
                cells.append(fmt(chip, t, c, v))
            lines.append("    " + " & ".join(cells) + " \\\\")
        lines.append("    \\midrule")

    # Positive and negative control rows
    for extra in ["PC", "NC"]:
        cells = [f"\\multicolumn{{2}}{{l}}{{{_CONTROL_DISPLAY[extra]}}}"]
        for chip in chips:
            v = get(chip, (extra, ''))
            cells.append(f"{v:,}" if v else "--")
        lines.append("    " + " & ".join(cells) + " \\\\")

    lines.append("    \\midrule")
    cells = ["\\multicolumn{2}{l}{\\textbf{Total}}"]
    for chip in chips:
        total = get(chip, ('Total', ''))
        failed_sum = sum(get(chip, (rl, c)) for (ch, rl, c) in failed_cells if ch == chip)
        cells.append(f"{total - failed_sum:,}")
    lines.append("    " + " & ".join(cells) + " \\\\")

    if n_timesteps:
        lines.append("    \\midrule")
        cells = ["\\multicolumn{2}{l}{Number of Time Steps}"]
        for chip in chips:
            cells.append(f"{n_timesteps.get(chip, 0):,}")
        lines.append("    " + " & ".join(cells) + " \\\\")

    lines.append("    \\bottomrule")
    lines.append("    \\end{tabular}")

    if special_notes or failed_cells:
        notes = []
        if special_notes:
            notes.append(_FOOTNOTE_SPECIAL)
        if failed_cells:
            notes.append(_FOOTNOTE_FAILED)
        lines.append("")
        lines.append("    \\vspace{1ex}")
        lines.append("    \\raggedright")
        lines.append("    \\footnotesize " + " \\\\\n    ".join(notes))

    lines.append("\\end{table}")
    return "\n".join(lines)


# Version 1: plain, no strikethrough -- matches the reference style/wording exactly.
latex_table_plain = to_latex_profile_table_transposed(
    profile,
    caption="Per-well active-pixel curve counts across the six Lacewing eLAMP chips. "
            "The data is grouped by target and concentration alongside the respective "
            "positive and negative control wells.",
    label="tab:final_6chip_profile",
)
print(latex_table_plain)


\begin{table}[htbp]
    \centering
    \caption{Per-well active-pixel curve counts across the six Lacewing eLAMP chips. The data is grouped by target and concentration alongside the respective positive and negative control wells.}
    \label{tab:final_6chip_profile}
    \small
    \begin{tabular}{@{}llrrrrrr@{}}
    \toprule
    \textbf{Target} & \textbf{Conc.} & \textbf{Chip 01} & \textbf{Chip 02} & \textbf{Chip 03} & \textbf{Chip 04} & \textbf{Chip 05} & \textbf{Chip 06} \\
    \midrule
    \multirow{3}{*}{IAV} & $10^4$ & 1,914 & -- & 1,946 & -- & -- & 1,890 \\
     & $10^5$ & 1,871 & 1,930 & -- & -- & 1,955 & -- \\
     & $10^6$ & 1,942 & 1,949 & -- & 1,893 & -- & -- \\
    \midrule
    \multirow{3}{*}{IBV} & $10^4$ & 1,807 & 1,918 & -- & 1,852 & -- & 1,827 \\
     & $10^5$ & -- & 1,881 & 1,872 & -- & -- & 1,936 \\
     & $10^6$ & -- & 1,924 & 1,892 & -- & 1,919 & -- \\
    \midrule
    \multirow{3}{*}{KP} & $10^4$ & -- & 1,756 & 1,868 & -- & 1,943 & -- \\
     & $10^5$ & 1,713 & --

In [6]:
# Version 2: same style, with non-PC/NC cells config.LOFO_EXCLUDE_WELL_MAPPING[GROUP_NAME]
# drops for LOFO training shown as "--" + a footnote marker (PC/NC stay unmarked either way).
excluded_cells = build_lofo_excluded_cells(GROUP_NAME)
failed_cells = {key for key in excluded_cells if key[1] not in ("PC", "NC")}
special_notes = {("DDM_06", "hAdV", 10_000): "$^*$"}


def chip_n_timesteps(dataset_name):
    d = joblib.load(os.path.join(EXP_FOLDER, dataset_name, config.TRAINING_DATA_PATH))
    return len(d["timestamps"])


n_timesteps = {chip_label(name): chip_n_timesteps(name) for name in DATASETS}

latex_table_lofo = to_latex_profile_table_transposed(
    profile,
    caption="Per-well active-pixel curve counts across the six Lacewing eLAMP chips.",
    label="tab:elamp_dataset",
    failed_cells=failed_cells,
    special_notes=special_notes,
    n_timesteps=n_timesteps,
    placement="H",
)
print(latex_table_lofo)


\begin{table}[H]
    \centering
    \caption{Per-well active-pixel curve counts across the six Lacewing eLAMP chips.}
    \label{tab:elamp_dataset}
    \small
    \begin{tabular}{@{}llrrrrrr@{}}
    \toprule
    \textbf{Target} & \textbf{Conc.} & \textbf{Chip 01} & \textbf{Chip 02} & \textbf{Chip 03} & \textbf{Chip 04} & \textbf{Chip 05} & \textbf{Chip 06} \\
    \midrule
    \multirow{3}{*}{IAV} & $10^4$ & 1,914 & -- & 1,946 & -- & -- & 1,890 \\
     & $10^5$ & 1,871 & 1,930 & -- & -- & 1,955 & -- \\
     & $10^6$ & 1,942 & 1,949 & -- & 1,893 & -- & -- \\
    \midrule
    \multirow{3}{*}{IBV} & $10^4$ & 1,807 & 1,918 & -- & 1,852 & -- & --$^{**}$ \\
     & $10^5$ & -- & 1,881 & 1,872 & -- & -- & 1,936 \\
     & $10^6$ & -- & 1,924 & 1,892 & -- & 1,919 & -- \\
    \midrule
    \multirow{3}{*}{KP} & $10^4$ & -- & 1,756 & 1,868 & -- & 1,943 & -- \\
     & $10^5$ & 1,713 & -- & 1,668 & 1,678 & -- & -- \\
     & $10^6$ & -- & -- & 1,904 & 1,825 & -- & 1,864 \\
    \midrule
    \multirow{3}

In [7]:
# out_path = os.path.join(_MAIN_DIR, "notebooks", "final_6chip_dataset_profile_table.tex")
# with open(out_path, "w") as f:
#     f.write(latex_table + "\n")
# print(f"Saved -> {out_path}")

